# 7장. 디코딩 방법 비교하기

6장의 모델을 더 오래 학습시킨 다음, 같은 모델로 탐욕 디코딩, 빔 탐색, 온도, top-k, top-p, 반복 벌점을 직접 구현해 비교한다. 사람이 쓴 글을 기준으로 그럴듯함, 반복, 다양성을 재고, 끝으로 정해진 형식으로만 답하게 하는 방법을 만든다.

**준비:** 한국어 위키백과 덤프의 첫 조각 `kowiki-latest-pages-articles1.xml-p1p82407.bz2`(4장과 같은 파일)를 왼쪽 파일 창에 올리고, 코랩에서는 `!pip install tokenizers`를 먼저 실행한다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 16분쯤 걸렸다.

## 7.1 모델과, 사람이 쓴 글로 정한 기준

In [ ]:
import bz2, re, math, time, os, sys, random, json, platform
from collections import Counter
from xml.etree.ElementTree import iterparse
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
os.environ["RAYON_NUM_THREADS"] = "2"
import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
torch.set_num_threads(2)

pages = []
with bz2.open("kowiki-latest-pages-articles1.xml-p1p82407.bz2", "rb") as f:
    for _, el in iterparse(f, events=("end",)):
        if el.tag.endswith("}page"):
            if el.findtext("{*}ns") == "0" and el.find("{*}redirect") is None:
                pages.append((el.findtext("{*}title"), el.findtext("{*}revision/{*}text") or ""))
            el.clear()
print(len(pages))
print(sys.version.split()[0], torch.__version__, tokenizers.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
# 4장의 clean()을 그대로 쓴다. 이번에는 문장으로 나누지 않고 문단을 이어 문서 하나를 글 하나로 만든다
def strip_nested(text, left, right):
    """가장 안쪽의 left ... right를 지우기를 더 지울 것이 없을 때까지 한다(틀 {{ }}, 표 {| |})."""
    inner = re.compile(re.escape(left) + r"(?:(?!" + re.escape(left) + r").)*?" + re.escape(right), re.S)
    while True:
        new = inner.sub("", text)
        if new == text:
            return new
        text = new

def clean(wiki):
    """위키 문법을 걷어 내고 문단(줄) 목록을 돌려준다."""
    t = re.sub(r"<!--.*?-->", "", wiki, flags=re.S)                       # 주석
    t = re.sub(r"<ref[^>]*/>", "", t)                                      # 각주
    t = re.sub(r"<ref[^>]*>.*?</ref>", "", t, flags=re.S)
    t = strip_nested(t, "{{", "}}")                                        # 틀(정보 상자 등)
    t = strip_nested(t, "{|", "|}")                                        # 표
    t = re.sub(r"<(math|gallery|timeline|score|syntaxhighlight|source|pre|code)[^>]*>.*?</\1>", "", t, flags=re.S | re.I)
    t = re.sub(r"<[^>]+>", "", t)                                          # 남은 HTML 태그
    t = re.sub(r"\[\[(파일|File|그림|Image|분류|Category):[^\[\]]*(\[\[[^\]]*\]\][^\[\]]*)*\]\]", "", t, flags=re.I)
    t = re.sub(r"\[\[([^\]|]*)\|([^\]]*)\]\]", r"\2", t)                   # [[문서|보이는 글]] → 보이는 글
    t = re.sub(r"\[\[([^\]]*)\]\]", r"\1", t)                              # [[문서]] → 문서
    t = re.sub(r"\[https?://[^\s\]]+ ([^\]]*)\]", r"\1", t)                # [주소 글] → 글
    t = re.sub(r"\[https?://[^\]]*\]", "", t)
    t = re.sub(r"'{2,}", "", t)                                            # 굵게·기울임
    t = re.sub(r"__[A-Z]+__", "", t)
    out = []
    for line in t.split("\n"):
        line = line.strip()
        if not line or line.startswith(("=", "|", "!", "{", "}")):          # 제목, 표의 나머지
            continue
        line = re.sub(r"^[*#:;]+\s*", "", line)                            # 목록 기호
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            out.append(line)
    return out

def sentences(paragraph):
    return [s.strip() for s in re.split(r"(?<=[.!?。])\s+", paragraph) if s.strip()]

hangul = lambda s: sum("가" <= ch <= "힣" for ch in s)
keep = lambda p: len(p) >= 20 and hangul(p) / max(1, sum(ch.isalpha() for ch in p)) >= 0.5
docs = [" ".join(p for p in clean(text) if keep(p)) for _, text in pages]
held = set(random.Random(0).sample(range(len(pages)), len(pages) // 10))   # 4·5장과 같은 검증용 문서
train_docs = [d for i, d in enumerate(docs) if i not in held and d]
val_docs = [d for i, d in enumerate(docs) if i in held and d]
print(len(train_docs), len(val_docs), sum(map(len, train_docs)), sum(map(len, val_docs)))

In [ ]:
pick = random.Random(0)
paras = [p for d in train_docs for p in d.split(". ")]
letters = {ch for ch, n in Counter(ch for p in pick.sample(paras, 300000) for ch in p).items() if n >= 5}
UNK_CH, SP = "□", "▁"
to_text = lambda t: "".join(ch if ch in letters else UNK_CH for ch in t).replace(" ", SP)
tok = Tokenizer(models.BPE(unk_token=UNK_CH))
tok.pre_tokenizer = pre_tokenizers.Split(SP, behavior="merged_with_next")
tok.train_from_iterator([to_text(p) for p in pick.sample(paras, 200000)], trainers.BpeTrainer(
    vocab_size=8000, show_progress=False, initial_alphabet=sorted(letters - {" "} | {SP, UNK_CH}),
    special_tokens=["<패딩>", "<시작>", "<끝>"]))
PAD, START, END, VOCAB = 0, 1, 2, tok.get_vocab_size()

LENGTH = 512                                               # 모델이 한 번에 읽는 토큰 수
def chunks(texts):
    """문서마다 <시작>·<끝>을 붙여 한 줄로 이은 다음, LENGTH + 1개씩 자른다(마지막 1개는 정답으로만 쓴다)."""
    stream = [t for e in tok.encode_batch([to_text(x) for x in texts]) for t in [START] + e.ids + [END]]
    n = len(stream) // (LENGTH + 1)
    return torch.tensor(stream[:n * (LENGTH + 1)]).view(n, LENGTH + 1)

train_rows = chunks(pick.sample(train_docs, len(train_docs)))
val_rows = chunks(val_docs)
show = lambda ids: tok.decode([i for i in ids if i > END]).replace(" ", "").replace(SP, " ")   # 토큰 번호를 글로
print(VOCAB, len(letters), tuple(train_rows.shape), tuple(val_rows.shape))
print(show(val_rows[0, :40].tolist()))

In [ ]:
def rotate(x, pos):
    """3장 3.2절의 RoPE. 이번에는 위치 번호 pos(배치, 길이)를 받아, 그 위치의 각도만큼 돌린다."""
    hd = x.shape[-1]
    theta = 10000.0 ** (-torch.arange(0, hd, 2).float() / hd)
    angle = pos[:, None, :, None].float() * theta                     # (배치, 1, 길이, hd/2)
    c, s = torch.cos(angle), torch.sin(angle)
    a, b = x[..., 0::2], x[..., 1::2]
    return torch.stack([a * c - b * s, a * s + b * c], dim=-1).flatten(-2)

class Attention(nn.Module):
    """RoPE를 넣은 셀프 어텐션. kv_heads가 heads보다 작으면 그룹 질의 어텐션(3장 3.5절)이다."""
    def __init__(self, dim, heads, kv_heads):
        super().__init__()
        self.heads, self.kv_heads, self.hd = heads, kv_heads, dim // heads
        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(dim, kv_heads * self.hd)
        self.to_v = nn.Linear(dim, kv_heads * self.hd)
        self.out = nn.Linear(dim, dim)

    def forward(self, x, pos, allow, cache=None):
        B, L, D = x.shape
        q = rotate(self.to_q(x).view(B, L, self.heads, self.hd).transpose(1, 2), pos)
        k = rotate(self.to_k(x).view(B, L, self.kv_heads, self.hd).transpose(1, 2), pos)
        v = self.to_v(x).view(B, L, self.kv_heads, self.hd).transpose(1, 2)
        if cache is not None:                                 # 6.2절: 앞에서 만든 키·값 뒤에 이번 것을 붙여 저장한다
            if "k" in cache:
                k, v = torch.cat([cache["k"], k], dim=2), torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=allow, enable_gqa=self.kv_heads < self.heads)
        return self.out(a.transpose(1, 2).reshape(B, L, D))

class Block(nn.Module):
    def __init__(self, dim, heads, kv_heads):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attend = Attention(dim, heads, kv_heads)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x, pos, allow, cache=None):
        x = x + self.attend(self.norm1(x), pos, allow, cache)
        return x + self.mlp(self.norm2(x))

class LanguageModel(nn.Module):
    def __init__(self, dim=192, layers=4, heads=6, kv_heads=6, size=8000):
        super().__init__()
        self.table = nn.Embedding(size, dim)                  # RoPE를 쓰므로 위치 표가 없다
        self.blocks = nn.ModuleList(Block(dim, heads, kv_heads) for _ in range(layers))
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, size)

    def forward(self, ids, pos=None, caches=None, real=None):
        """ids(배치, 길이). caches를 주면 블록마다 키·값을 쌓는다. real은 패딩이 아닌 토큰(6.5절)."""
        B, L = ids.shape
        before = caches[0]["real"].shape[1] if caches and "real" in caches[0] else 0   # 캐시에 이미 있는 토큰 수
        if pos is None:
            pos = torch.arange(before, before + L).expand(B, L)
        if real is None:
            real = torch.ones(B, L, dtype=torch.bool)
        if caches is not None:
            real = torch.cat([caches[0]["real"], real], dim=1) if before else real
            caches[0]["real"] = real
        q_at = torch.arange(before, before + L)                # 질의와 키가 줄에서 몇 번째 칸인지
        k_at = torch.arange(before + L)
        # 앞 칸만 본다(인과 마스크). 패딩 키는 보지 않되, 패딩 칸의 질의는 자기 자신만은 보게 해서 계산이 끊기지 않게 한다
        allow = (k_at[None, :] <= q_at[:, None]) & (real[:, None, :] | (k_at[None, :] == q_at[:, None]))
        x = self.table(ids)
        for i, block in enumerate(self.blocks):
            x = block(x, pos, allow[:, None], caches[i] if caches is not None else None)
        return self.head(self.norm(x))

count = lambda net: sum(p.numel() for p in net.parameters())
print(f"파라미터 {count(LanguageModel()):,}")

In [ ]:
def fit(net, X, lr=0.003, batch=16, seed=0):
    """6장과 같다: 1에폭, AdamW, 처음 10% 걸음 동안 올리고 코사인으로 줄인다."""
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.0)
    shuffle = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed))
    steps = math.ceil(len(X) / batch)
    warm = max(1, steps // 10)
    net.train()
    for step in range(1, steps + 1):
        ids = X[shuffle[(step - 1) * batch:step * batch]]
        for group in optimizer.param_groups:
            group["lr"] = lr * min(1, step / warm) * 0.5 * (1 + math.cos(math.pi * step / steps))
        out = net(ids[:, :-1])
        loss = F.cross_entropy(out.reshape(-1, VOCAB), ids[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    net.eval()
    return net

def loss_of(net, X):
    with torch.inference_mode():
        return sum(F.cross_entropy(net(X[k:k + 8, :-1]).reshape(-1, VOCAB), X[k:k + 8, 1:].reshape(-1), reduction="sum").item()
                   for k in range(0, len(X), 8)) / X[:, 1:].numel()

torch.manual_seed(0)
lap = time.perf_counter()
net = fit(LanguageModel(), train_rows[:16_000_000 // LENGTH])     # 6장의 두 배인 토큰 1,600만 개
print(f"학습 {(time.perf_counter() - lap) / 60:.1f}분, 검증 손실 {loss_of(net, val_rows[:200]):.4f}")

In [ ]:
N_PROMPT, P_LEN, G_LEN = 200, 64, 64
prompts = val_rows[:N_PROMPT, :P_LEN]                       # 검증용 줄 200개의 앞 64토큰
human = val_rows[:N_PROMPT, P_LEN:P_LEN + G_LEN]            # 그 뒤에 실제로 이어진 64토큰(사람이 쓴 글)

def nll(cont):
    """프롬프트 뒤에 cont가 이어질 때, 모델이 cont의 토큰마다 매긴 음의 로그 확률의 평균(문장마다)."""
    ids = torch.cat([prompts, cont], dim=1)
    with torch.inference_mode():
        logp = torch.cat([F.log_softmax(net(ids[k:k + 50, :-1]), dim=-1) for k in range(0, len(ids), 50)])
    got = logp[:, P_LEN - 1:].gather(2, cont[:, :, None])[:, :, 0]
    return -got.mean(dim=1)

def grams(seq, n):
    return [tuple(seq[i:i + n]) for i in range(len(seq) - n + 1)]

def rep4(seq):
    """4-그램 가운데 같은 글 안에서 앞에 이미 나온 것의 비율(반복)."""
    seen, again = set(), 0
    for g in grams(seq, 4):
        again += g in seen
        seen.add(g)
    return again / len(grams(seq, 4))

distinct2 = lambda seq: len(set(grams(seq, 2))) / len(grams(seq, 2))     # 서로 다른 2-그램의 비율(다양성)
PERIOD = torch.tensor([tok.id_to_token(i).endswith(".") for i in range(VOCAB)])
def score(cont):
    """문장 200개의 평균: 퍼플렉서티(모델이 본 그럴듯함), 4-그램 반복률, distinct-2, 마침표로 끝나는 토큰의 비율."""
    rows = cont.tolist()
    return {"ppl": math.exp(nll(cont).mean().item()), "rep4": float(np.mean([rep4(r) for r in rows])),
            "dist2": float(np.mean([distinct2(r) for r in rows])), "period": PERIOD[cont].float().mean().item()}

results = {"사람": score(human)}
print("사람이 쓴 글:", {k: round(v, 4) for k, v in results["사람"].items()})

## 7.2 탐욕 디코딩과 빔 탐색

In [ ]:
def generate(net, prompts, n, choose):
    """6장의 KV 캐시로 n토큰을 만든다. choose(로짓, 지금까지의 토큰)가 걸음마다 다음 토큰을 고른다."""
    caches = [{} for _ in net.blocks]
    ids = prompts
    with torch.inference_mode():
        last = net(ids, caches=caches)[:, -1]
        for _ in range(n):
            nxt = choose(last, ids)
            ids = torch.cat([ids, nxt], dim=1)
            last = net(nxt, caches=caches)[:, -1]
    return ids[:, prompts.shape[1]:]

greedy = lambda logits, ids: logits.argmax(dim=-1, keepdim=True)

def beam_search(net, prompts, n, width=4):
    """문장마다 후보 width개를 들고 가며, 로그 확률의 합이 가장 큰 width개만 남긴다."""
    B, V = prompts.shape[0], VOCAB
    caches = [{} for _ in net.blocks]
    with torch.inference_mode():
        logp = F.log_softmax(net(prompts.repeat_interleave(width, dim=0), caches=caches)[:, -1], dim=-1)
        total = torch.full((B, width), float("-inf"))
        total[:, 0] = 0.0                                     # 처음에는 후보가 하나뿐이다(모두 같은 프롬프트)
        seqs = torch.zeros(B * width, 0, dtype=torch.long)
        for _ in range(n):
            cand = (total.view(B * width, 1) + logp).view(B, width * V)     # 후보마다 다음 토큰 V개를 이어 본다
            top = cand.topk(width, dim=-1)
            total = top.values
            src = (torch.arange(B)[:, None] * width + top.indices // V).view(-1)   # 어느 후보에서 이어졌는가
            seqs = torch.cat([seqs[src], (top.indices % V).view(-1, 1)], dim=1)
            for c in caches:                                  # 캐시도 살아남은 후보의 순서대로 다시 고른다
                c["k"], c["v"] = c["k"][src], c["v"][src]
            caches[0]["real"] = caches[0]["real"][src]
            logp = F.log_softmax(net(seqs[:, -1:], caches=caches)[:, -1], dim=-1)
    return seqs.view(B, width, n)[:, 0]                       # topk는 큰 순서이므로 0번이 가장 높은 후보다

made = {"탐욕": generate(net, prompts, G_LEN, greedy), "빔 4": beam_search(net, prompts, G_LEN, 4)}
for name in made:
    results[name] = score(made[name])
    print(f"{name}:", {k: round(v, 4) for k, v in results[name].items()})
i = 3
print("프롬프트:", show(prompts[i].tolist())[-60:])
for name, cont in [("사람", human), ("탐욕", made["탐욕"]), ("빔 4", made["빔 4"])]:
    print(f"{name}:", show(cont[i].tolist()))

## 7.3 온도, top-k, top-p

In [ ]:
def sampler(temp=1.0, top_k=None, top_p=None, penalty=1.0, seed=0, sizes=None):
    """확률 분포에서 뽑는다. 온도로 나누고, top-k나 top-p로 후보를 자르고, 반복 벌점(7.4절)을 준다."""
    gen = torch.Generator().manual_seed(seed)
    def choose(logits, ids):
        logits = logits.clone()
        if penalty != 1.0:                                    # 이미 나온 토큰: 양수 로짓은 나누고 음수 로짓은 곱한다
            seen = torch.zeros_like(logits, dtype=torch.bool).scatter_(1, ids, True)
            logits = torch.where(seen, torch.where(logits > 0, logits / penalty, logits * penalty), logits)
        if temp == 0:
            return logits.argmax(dim=-1, keepdim=True)
        logits = logits / temp
        if top_k is not None:
            kth = logits.topk(top_k, dim=-1).values[:, -1:]
            logits = logits.masked_fill(logits < kth, float("-inf"))
        if top_p is not None:                                 # 확률이 큰 순서로 더해 top_p를 처음 넘는 데까지만 남긴다
            p, order = torch.softmax(logits, dim=-1).sort(dim=-1, descending=True)
            drop = (p.cumsum(dim=-1) - p) >= top_p
            if sizes is not None:
                sizes.extend((~drop).sum(dim=-1).tolist())
            logits = logits.masked_fill(drop.scatter(1, order, drop), float("-inf"))
        return torch.multinomial(torch.softmax(logits, dim=-1), 1, generator=gen)
    return choose

CONFIGS = {"온도 0.7": dict(temp=0.7), "온도 1.0": dict(temp=1.0), "온도 1.3": dict(temp=1.3),
           "top-k 10": dict(top_k=10), "top-k 40": dict(top_k=40),
           "top-p 0.5": dict(top_p=0.5), "top-p 0.7": dict(top_p=0.7), "top-p 0.9": dict(top_p=0.9), "top-p 0.95": dict(top_p=0.95)}
spread = {}
for name, cfg in CONFIGS.items():
    runs = [score(generate(net, prompts, G_LEN, sampler(**cfg, seed=s))) for s in range(3)]
    results[name] = {k: float(np.median([r[k] for r in runs])) for k in runs[0]}
    spread[name] = {k: float(max(r[k] for r in runs) - min(r[k] for r in runs)) for k in runs[0]}
    print(f"{name:<10}", "  ".join(f"{k} {results[name][k]:.4f}(±{spread[name][k] / 2:.4f})" for k in runs[0]), flush=True)

In [ ]:
# top-p 0.9로 만들 때, 걸음마다 남은 후보 수를 센다
sizes = []
generate(net, prompts, G_LEN, sampler(top_p=0.9, seed=0, sizes=sizes))
sizes = np.array(sizes)
print(f"걸음 {len(sizes):,}개: 남은 후보 중앙값 {np.median(sizes):.0f}, 10% {np.percentile(sizes, 10):.0f}, 90% {np.percentile(sizes, 90):.0f}, "
      f"최대 {sizes.max()}, 1개뿐 {np.mean(sizes == 1):.1%}, 40개 넘음 {np.mean(sizes > 40):.1%}")
# 프롬프트 200개의 첫 걸음 분포 가운데, 가장 높은 확률이 가장 큰 것(뾰족한 분포)과 가장 작은 것(평평한 분포)을 그림으로 남긴다
with torch.inference_mode():
    first_step = torch.softmax(net(prompts)[:, -1], dim=-1)
top1 = first_step.max(dim=-1).values
pick_rows = [int(top1.argmax()), int(top1.argmin())]
probe = first_step[pick_rows]
for r, row in enumerate(pick_rows):
    p, order = probe[r].sort(descending=True)
    print(f"프롬프트 {row}: 가장 높은 확률 {p[0]:.3f}, top-p 0.9에 드는 후보 {int(((p.cumsum(0) - p) < 0.9).sum())}개, "
          f"상위 5개 {[tok.id_to_token(int(t)) for t in order[:5]]}")

In [ ]:
# 프롬프트 하나에 시드 10개로 만들면 지표가 얼마나 달라지는가(top-p 0.9)
one = prompts[3:4]
single = []
for s in range(10):
    cont = generate(net, one, G_LEN, sampler(top_p=0.9, seed=s))[0].tolist()
    single.append((rep4(cont), distinct2(cont)))
single = np.array(single)
print(f"프롬프트 하나, 시드 10개: 4-그램 반복률 {single[:, 0].min():.3f}~{single[:, 0].max():.3f}, distinct-2 {single[:, 1].min():.3f}~{single[:, 1].max():.3f}")
print(f"프롬프트 200개의 평균, 시드 3개: 4-그램 반복률 범위 {spread['top-p 0.9']['rep4']:.4f}, distinct-2 범위 {spread['top-p 0.9']['dist2']:.4f}")
for s in range(3):
    print(f"시드 {s}:", show(generate(net, one, 30, sampler(top_p=0.9, seed=s))[0].tolist()))

## 7.4 반복 벌점

In [ ]:
PENALTY = {"탐욕+벌점 1.2": dict(temp=0, penalty=1.2), "탐욕+벌점 1.5": dict(temp=0, penalty=1.5),
           "top-p 0.9+벌점 1.2": dict(top_p=0.9, penalty=1.2), "top-p 0.9+벌점 1.5": dict(top_p=0.9, penalty=1.5)}
for name, cfg in PENALTY.items():
    runs = [score(generate(net, prompts, G_LEN, sampler(**cfg, seed=s))) for s in range(1 if cfg.get("temp") == 0 else 3)]
    results[name] = {k: float(np.median([r[k] for r in runs])) for k in runs[0]}
    print(f"{name:<16}", "  ".join(f"{k} {results[name][k]:.4f}" for k in runs[0]), flush=True)
cont = generate(net, prompts[3:4], 40, sampler(temp=0, penalty=1.5))
print("탐욕+벌점 1.5:", show(cont[0].tolist()))
# 사람이 쓴 글에서 앞에 이미 나온 토큰이 다시 나오는 비율. 벌점은 이 토큰들을 모두 깎는다
again = [np.mean([t in set(p + h[:j]) for j, t in enumerate(h)]) for p, h in zip(prompts.tolist(), human.tolist())]
print(f"사람이 쓴 글의 토큰 가운데 프롬프트나 앞에서 이미 나온 토큰: {np.mean(again):.1%}")

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter, NullFormatter
for path in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf"))
             + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(path)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 8, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"

def family(name):
    return (RED if name in ("탐욕", "빔 4") else ORANGE if name.startswith("온도") else BLUE if name.startswith("top-k")
            else GREEN if name.startswith("top-p") and "벌점" not in name else PURPLE if "벌점" in name else "#111827")
order = sorted(results, key=lambda n: results[n]["ppl"])          # 퍼플렉서티가 낮은 것부터 위에서 아래로
fig, axes = plt.subplots(1, 2, figsize=(5.2, 4.7), dpi=260, sharey=True, gridspec_kw={"width_ratios": [1.25, 1]})
y = np.arange(len(order))[::-1]
for ax, key, label, scale in [(axes[0], "ppl", "퍼플렉서티(로그 눈금)", "log"), (axes[1], "rep4", "4-그램 반복률(%)", "linear")]:
    vals = [results[n][key] * (100 if key == "rep4" else 1) for n in order]
    ax.scatter(vals, y, c=[family(n) for n in order], s=[150 if n == "사람" else 32 for n in order],
               marker="o", zorder=3, edgecolors="white", linewidths=0.6)
    ax.axvline(results["사람"][key] * (100 if key == "rep4" else 1), color="#111827", lw=1, ls=":", zorder=1)
    ax.set_xscale(scale); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    ax.set_xlabel(label); ax.grid(axis="x", color="#eef0f3", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
    for yi, v, n in zip(y, vals, order):                     # 값은 점의 오른쪽에 적는다
        ax.annotate(f"{v:.1f}" if key == "rep4" else f"{v:.0f}" if v >= 10 else f"{v:.1f}", (v, yi),
                    xytext=(9 if n == "사람" else 5, 0), textcoords="offset points", va="center", fontsize=8, color="#4b5563")
axes[0].set_xlim(3, 1500); axes[1].set_xlim(-3, 76)                # 점 오른쪽의 값이 칸 안에 들게 여유를 둔다
axes[0].set_yticks(y); axes[0].set_yticklabels(order, fontsize=8)
for t, n in zip(axes[0].get_yticklabels(), order):
    t.set_color(family(n))
axes[0].set_title("① 모델이 본 그럴듯함\n(점선은 사람이 쓴 글)", loc="left", fontsize=8.4, color="#374151")
axes[1].set_title("② 같은 말을\n거듭 쓰는 정도", loc="left", fontsize=8.4, color="#374151")
fig.tight_layout(); fig.savefig("fig_7_map.png"); plt.close(fig)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 4.5), dpi=260)
for r, ax in enumerate(axes):
    p, order = probe[r].sort(descending=True)
    n_top = int(((p.cumsum(0) - p) < 0.9).sum())              # top-p 0.9에 드는 후보 수(분포 전체에서)
    p = p[:20].numpy()
    inside = (np.cumsum(p) - p) < 0.9
    colors = [GREEN if i == 0 else (BLUE if inside[i] else "#D1D5DB") for i in range(20)]
    ax.bar(range(20), p, color=colors, width=0.75)
    ax.axvline(9.5, color=ORANGE, ls="--", lw=1.1)
    ax.text(9.75, p.max() * 0.92, "top-k 10의 경계", color=ORANGE, fontsize=8)
    ax.set_xlim(-0.7, 19.7)
    ax.set_xticks(range(20))
    ax.set_xticklabels([tok.id_to_token(int(t)) for t in order[:20]], fontsize=8, rotation=60, ha="right", rotation_mode="anchor")
    ax.tick_params(axis="y", labelsize=8)
    ax.set_ylabel("확률")
    ax.set_title(f"{'①②'[r]} 프롬프트 {pick_rows[r]}의 다음 토큰 상위 20개\n초록은 탐욕이 고르는 토큰, 파랑까지 top-p 0.9의 후보({n_top}개)",
                 loc="left", fontsize=8.4, color="#374151")
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(h_pad=1.2); fig.savefig("fig_7_filter.png"); plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(5.2, 3.2), dpi=260)
ax = axes[0]
ax.hist(sizes, bins=np.logspace(0, np.log10(sizes.max() + 1), 30), color=GREEN, alpha=0.85)
ax.axvline(40, color=BLUE, ls="--", lw=1.1); ax.text(45, ax.get_ylim()[1] * 0.9, "top-k 40", color=BLUE, fontsize=8)
ax.set_xscale("log"); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlabel("top-p 0.9에 남은 후보 수\n(로그 눈금)"); ax.set_ylabel("걸음 수")
ax.set_title("① 걸음마다 남은 후보 수", loc="left", fontsize=8.4, color="#374151")
ax = axes[1]
for names, color, label in [(["온도 0.7", "온도 1.0", "온도 1.3"], ORANGE, "온도"), (["top-k 10", "top-k 40"], BLUE, "top-k"),
                            (["top-p 0.5", "top-p 0.7", "top-p 0.9", "top-p 0.95"], GREEN, "top-p")]:
    xs = [results[n]["dist2"] for n in names]; ys = [results[n]["ppl"] for n in names]
    ax.plot(xs, ys, "-o", color=color, ms=3.5, lw=1.2, label=label)
ax.plot(results["사람"]["dist2"], results["사람"]["ppl"], "*", color="#111827", ms=11, label="사람")
ax.set_xlim(0.92, 1.0); ax.set_xticks([0.92, 0.94, 0.96, 0.98, 1.0])
ax.text(0.9215, 560, f"탐욕은 distinct-2 {results['탐욕']['dist2']:.2f},\n퍼플렉서티 {results['탐욕']['ppl']:.1f}로 칸 밖",
        fontsize=8, color=RED, va="top")
ax.set_yscale("log"); ax.set_ylim(12, 650); ax.set_yticks([20, 50, 100, 200, 500])
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}")); ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel("distinct-2\n(높을수록 다양)"); ax.set_ylabel("퍼플렉서티(로그 눈금)")
ax.set_title("② 값을 바꾸며 그린 길", loc="left", fontsize=8.4, color="#374151")
ax.legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0, 0.1), handlelength=1.6, borderaxespad=0.2)
for a in axes:
    a.grid(color="#eef0f3", lw=0.6); a.spines[["top", "right"]].set_visible(False)
    a.tick_params(labelsize=8)
fig.tight_layout(w_pad=1.0); fig.savefig("fig_7_sweep.png"); plt.close(fig)

## 7.5 정해진 형식으로만 답하게 하기

In [ ]:
# 검증용 문서 가운데 '이름(한자, 1936년 12월 19일' 꼴로 시작하는 인물 문서를 찾아, 날짜 앞까지를 프롬프트로 쓴다
DATE = re.compile(r"^([^()]{1,30}\([^()]{0,40}?,) ?(\d{4})년 (\d{1,2})월 (\d{1,2})일")
cases = [(m.group(1), (int(m.group(2)), int(m.group(3)), int(m.group(4)))) for d in val_docs if (m := DATE.match(d))]
print("날짜로 시작하는 인물 문서:", len(cases), "예:", cases[0])

# 형식: 빈칸 하나(있어도 없어도 된다), 숫자 4개, '년', 빈칸, 숫자 1~2개, '월', 빈칸, 숫자 1~2개, '일'
DIGIT = "0123456789"
SPEC = [(" ", 0, 1), (DIGIT, 4, 4), ("년", 1, 1), (" ", 1, 1), (DIGIT, 1, 2), ("월", 1, 1), (" ", 1, 1), (DIGIT, 1, 2), ("일", 1, 1)]
def closure(states):
    """칸 j에서 최소 횟수를 채웠으면 다음 칸으로 넘어간 상태도 함께 둔다."""
    out, stack = set(states), list(states)
    while stack:
        j, c = stack.pop()
        if j < len(SPEC) and c >= SPEC[j][1] and (j + 1, 0) not in out:
            out.add((j + 1, 0)); stack.append((j + 1, 0))
    return out
def advance(states, text):
    for ch in text:
        states = closure({(j, c + 1) for j, c in states if j < len(SPEC) and ch in SPEC[j][0] and c < SPEC[j][2]})
        if not states:
            break
    return states
START_STATE = closure({(0, 0)})
done = lambda states: (len(SPEC), 0) in states
PIECE = [tok.id_to_token(i).replace(SP, " ") if i > END else "" for i in range(VOCAB)]

def answer(prefix, constrained, choose=None, n=20):
    """프롬프트 뒤에 토큰을 만든다. constrained면 걸음마다 형식에 맞게 이어지는 토큰만 남긴다."""
    ids = torch.tensor([[START] + tok.encode(to_text(prefix)).ids])
    caches, states, text, mass = [{} for _ in net.blocks], START_STATE, "", []
    space_only = torch.tensor([PIECE[i] == " " for i in range(VOCAB)])     # 빈칸 하나뿐인 토큰 '▁'(날짜가 아닌 글로도 이어진다)
    with torch.inference_mode():
        last = net(ids, caches=caches)[0, -1]
        for _ in range(n):
            if constrained:
                ok = torch.tensor([bool(PIECE[i]) and bool(advance(states, PIECE[i])) for i in range(VOCAB)])
                prob = torch.softmax(last, dim=-1)
                mass.append((prob[ok].sum().item(), prob[ok & ~space_only].sum().item()))   # 형식에 맞는 토큰의 확률('▁' 포함, 뺀 것)
                last = last.masked_fill(~ok, float("-inf"))
            nxt = choose(last[None], ids)[0] if choose else last.argmax()[None]
            text += PIECE[int(nxt)]
            if constrained:
                states = advance(states, PIECE[int(nxt)])
                if done(states):
                    break
            ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
            last = net(nxt.view(1, 1), caches=caches)[0, -1]
    return text, mass

ANSWER = re.compile(r"^ ?(\d{4})년 (\d{1,2})월 (\d{1,2})일")
fmt, masses, examples = {}, [], []
for name in ["탐욕", "top-p 0.9", "형식 강제(탐욕)"]:
    ok_form, ok_real, ok_year, ok_date, errors = 0, 0, 0, 0, []
    for k, (prefix, truth) in enumerate(cases):
        text, mass = answer(prefix, name.startswith("형식"), sampler(top_p=0.9, seed=k) if name == "top-p 0.9" else None)
        m = ANSWER.match(text)
        if name.startswith("형식"):
            masses.append(mass)
        if k < 3:
            examples.append((name, prefix[-20:], text, truth))
        if m:
            got = tuple(map(int, m.groups()))
            ok_form += 1; ok_year += got[0] == truth[0]; ok_date += got == truth
            ok_real += got[0] <= 2026 and 1 <= got[1] <= 12 and 1 <= got[2] <= 31      # 달력에 있을 수 있는 날짜인가
            errors.append(abs(got[0] - truth[0]))
    fmt[name] = (ok_form / len(cases), ok_real / len(cases), ok_year / len(cases), ok_date / len(cases), float(np.median(errors)) if errors else None)
    print(f"{name:<12} 형식이 맞음 {fmt[name][0]:.1%}  있을 수 있는 날짜 {fmt[name][1]:.1%}  연도가 맞음 {fmt[name][2]:.1%}  "
          f"날짜가 모두 맞음 {fmt[name][3]:.1%}  연도 차이의 중앙값 {fmt[name][4]}", flush=True)
first = [m[0][0] for m in masses]
first_digit = [m[0][1] for m in masses]
print(f"형식 강제: 첫 걸음에서 모델이 형식에 맞는 토큰에 준 확률 중앙값 {np.median(first):.3f}, 0.5 미만 {np.mean(np.array(first) < 0.5):.1%}")
print(f"  그 가운데 빈칸 하나뿐인 토큰 '▁'를 뺀 확률 중앙값 {np.median(first_digit):.3f}, 0.1 미만 {np.mean(np.array(first_digit) < 0.1):.1%}")
for e in examples:
    print(e)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 5.6), dpi=260, gridspec_kw={"height_ratios": [1.1, 1]})
ax = axes[0]
names = list(fmt)
x = np.arange(len(names))
for j, (label, color) in enumerate([("형식이 맞음", GRAY), ("있을 수 있는 날짜", ORANGE), ("연도가 맞음", BLUE), ("날짜가 모두 맞음", GREEN)]):
    vals = [fmt[n][j] * 100 for n in names]
    ax.bar(x + (j - 1.5) * 0.2, vals, 0.19, color=color, label=label)
    for xi, v in zip(x, vals):
        ax.text(xi + (j - 1.5) * 0.2, v + 1.5, f"{v:.1f}" if 0 < v < 1 else f"{v:.0f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=8); ax.set_ylabel("인물 문서의 %")
ax.set_ylim(0, 115); ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=2)
ax.set_title("① 날짜를 이어 쓰게 했을 때", loc="left", fontsize=8.4, color="#374151")
ax = axes[1]
ax.hist(first, bins=np.linspace(0, 1, 21), color=PURPLE, alpha=0.55, label=f"'▁' 포함(중앙값 {np.median(first):.2f})")
ax.hist(first_digit, bins=np.linspace(0, 1, 21), color=PURPLE, alpha=0.95, label=f"'▁' 뺀 것(중앙값 {np.median(first_digit):.2f})")
ax.legend(frameon=False, fontsize=8, loc="upper right")
ax.set_xlabel("첫 걸음에서 형식에 맞는 토큰의 확률 합"); ax.set_ylabel("인물 문서 수")
ax.set_title("② 모델이 형식에 준 확률", loc="left", fontsize=8.4, color="#374151")
for a in axes:
    a.grid(axis="y", color="#eef0f3", lw=0.6); a.spines[["top", "right"]].set_visible(False)
    a.tick_params(labelsize=8)
fig.tight_layout(h_pad=1.2); fig.savefig("fig_7_format.png"); plt.close(fig)